In [1]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import AlignIO
from Bio import SeqIO
import pprint
import pandas as pd

In [2]:
###after gfftobed and bedtools merge
path = "/mnt/pbil/" 


### illicium Verum

In [3]:
#gfftobed -m Illicium_verum_chromosomes_mRNA.gff > Illicium_verum_chromosomes_mRNA.bed
#bedtools sort -i Illicium_verum_chromosomes_mRNA.bed > Illicium_verum_chromosomes_mRNA.sort.bed
#bedtools merge -i Illicium_verum_chromosomes_mRNA.sort.bed -delim "," -c 4,6 -o collapse > Illicium_verum.mRNA.merge.bed

prot_file =path+"Infer_Anth/data/illicium/transcripts.bis.pep.fa"
bed_file = path+"Infer_Anth/data/illicium/Illicium_verum_chromosomes_mRNA.sort.bed"
bed_file_merge = path+"Infer_Anth/data/illicium/Illicium_verum.mRNA.merge.bed"

In [4]:
BED = pd.read_csv(bed_file, sep="\t", header=None, names=["chr", "start", "end", "id", "unk1", "strand"])

In [5]:
BED.head()

,chr,start,end,id,unk1,strand
0,Chr1,46802,47097,ID=ptg001517l.7;Parent=ptg001517l.7,.,+
1,Chr1,47211,206372,ID=ptg001517l.8;Parent=ptg001517l.8,.,-
2,Chr1,104645,869356,ID=ptg001517l.1;Parent=ptg001517l.1,.,-
3,Chr1,559704,1038455,ID=ptg001517l.9;Parent=ptg001517l.9,.,+
4,Chr1,1142792,1201269,ID=ptg001517l.2;Parent=ptg001517l.2,.,-


In [6]:
#ID=ptg001517l.7;Parent=ptg001517l.7
#id='ptg001517l.7'
records = SeqIO.parse(prot_file, "fasta")
sequences = []
for rec in records:
    seq = SeqRecord(rec.seq,id=rec.name.replace(".", "_") ,name="",description="")
    sequences.append(seq)

In [7]:
with open(path+"Infer_Anth/data/illicium/ive.all.pep.fasta", "w") as output_handle:
    SeqIO.write(sequences, output_handle, "fasta")

In [8]:
fasta_names = [seq.id for seq in sequences]

In [9]:
BED["names"]= BED.id.apply(lambda x: x.split(";")[0].split("=")[1].replace(".", "_"))

In [10]:
BED.head()

,chr,start,end,id,unk1,strand,names
0,Chr1,46802,47097,ID=ptg001517l.7;Parent=ptg001517l.7,.,+,ptg001517l_7
1,Chr1,47211,206372,ID=ptg001517l.8;Parent=ptg001517l.8,.,-,ptg001517l_8
2,Chr1,104645,869356,ID=ptg001517l.1;Parent=ptg001517l.1,.,-,ptg001517l_1
3,Chr1,559704,1038455,ID=ptg001517l.9;Parent=ptg001517l.9,.,+,ptg001517l_9
4,Chr1,1142792,1201269,ID=ptg001517l.2;Parent=ptg001517l.2,.,-,ptg001517l_2


In [11]:
BED_nosplicing = pd.DataFrame(BED[BED.names.isin(fasta_names)])
BED_nosplicing["seqnames"] = BED_nosplicing.chr
BED_nosplicing["ranges"] = [str(i)+"-"+str(j) for i,j in zip(BED_nosplicing["start"], BED_nosplicing["end"])]
BED_nosplicing["type"] = ["gene"]*len(BED_nosplicing)
BED_nosplicing["gene_id"] = BED_nosplicing["names"]
BED_nosplicing["phase"] = BED_nosplicing["strand"]
BED_nosplicing["source"] = ["figshare_sun"]*len(BED_nosplicing)

In [12]:
BED_nosplicing.head()

,chr,start,end,id,unk1,strand,names,seqnames,ranges,type,gene_id,phase,source
0,Chr1,46802,47097,ID=ptg001517l.7;Parent=ptg001517l.7,.,+,ptg001517l_7,Chr1,46802-47097,gene,ptg001517l_7,+,figshare_sun
1,Chr1,47211,206372,ID=ptg001517l.8;Parent=ptg001517l.8,.,-,ptg001517l_8,Chr1,47211-206372,gene,ptg001517l_8,-,figshare_sun
2,Chr1,104645,869356,ID=ptg001517l.1;Parent=ptg001517l.1,.,-,ptg001517l_1,Chr1,104645-869356,gene,ptg001517l_1,-,figshare_sun
3,Chr1,559704,1038455,ID=ptg001517l.9;Parent=ptg001517l.9,.,+,ptg001517l_9,Chr1,559704-1038455,gene,ptg001517l_9,+,figshare_sun
4,Chr1,1142792,1201269,ID=ptg001517l.2;Parent=ptg001517l.2,.,-,ptg001517l_2,Chr1,1142792-1201269,gene,ptg001517l_2,-,figshare_sun


In [13]:
##same thing but keeping the duplicated sequences
subset = pd.DataFrame(BED_nosplicing[["seqnames", "strand", "source", "type", "phase", "names", "gene_id", "start", "end"]])
subset.sort_values(by=["seqnames", "start"], inplace=True)
subset.to_csv(path+"Infer_Anth/data/illicium/illicium_verum_no_splicing.tsv", sep="\t", index=None)

### Annona cherimola

In [87]:
prot_file =path+"Infer_Anth/data/annona_cherimola/anche102_proteins_primaryTranscriptOnly_annot.fasta"
bed_file = path+"Infer_Anth/data/annona_cherimola/Anche102_gene.mRNA.bed"
bed_file_merge = path+"Infer_Anth/data/annona_cherimola/Anche102_gene.merge.bed"

In [88]:
#after merge
BED = pd.read_csv(bed_file, sep="\t", header=None, names=["chr", "start", "end", "id", "unk1", "strand"])

In [89]:
#ID=Anche102Chr1g0000030.1;Parent=Anche102Chr1g0000030;Alias=g24397.t1;Note="enhancer of rudimentary protein, putative"
#id='Anche102Chr1g0000030.1'
records = SeqIO.parse(prot_file, "fasta")
sequences = []
for rec in records:
    seq = SeqRecord(rec.seq,id=rec.name,name="",description="")
    sequences.append(seq)

In [84]:
with open(path+"Infer_Anth/data/annona_cherimola/ach.all.pep.fasta", "w") as output_handle:
    SeqIO.write(sequences, output_handle, "fasta")

In [90]:
fasta_names = [seq.id for seq in sequences]

In [91]:
BED["names"]= BED.id.apply(lambda x: x.split(";")[0].split("=")[1])

In [92]:
BED_nosplicing = pd.DataFrame(BED[BED.names.isin(fasta_names)])
BED_nosplicing["chr"] = BED_nosplicing.chr.apply(lambda x: x.split("102")[1])
BED_nosplicing["names"] = BED_nosplicing.names.apply(lambda x: str(x.split(".")[0])) 
BED_nosplicing["seqnames"] = BED_nosplicing.chr.apply(lambda x: "ach_" + str(x)) 
BED_nosplicing["ranges"] = [str(i)+"-"+str(j) for i,j in zip(BED_nosplicing["start"], BED_nosplicing["end"])]
BED_nosplicing["type"] = ["gene"]*len(BED_nosplicing)
BED_nosplicing["gene_id"] = BED_nosplicing["names"]
BED_nosplicing["phase"] = BED_nosplicing["strand"]
BED_nosplicing["source"] = ["IHSM"]*len(BED_nosplicing)

In [93]:
##same thing but keeping the duplicated sequences
subset = pd.DataFrame(BED_nosplicing[["seqnames", "strand", "source", "type", "phase", "names", "gene_id", "start", "end"]])
subset.sort_values(by=["seqnames", "start"], inplace=True)
subset.to_csv(path+"Infer_Anth/data/annona_cherimola/annona_cherimola_no_splicing.tsv", sep="\t", index=None)

### aristolochia contorta


In [3]:
prot_file =path+"Infer_Anth/data/aristo_contorta/62647-CDS-prot.fasta"
dedup_prot_file =path+"Infer_Anth/data/aristo_contorta/sort_aco.dedup.pep.fasta"
bed_file = path+"Infer_Anth/data/aristo_contorta/Aristolochia_contorta.CDS_id62647.merge.bed"
bed_file_gene = path+"Infer_Anth/data/aristo_contorta/Aristolochia_contorta.gene_id62647.bed"
bed_file_gene_merge = path+"Infer_Anth/data/aristo_contorta/Aristolochia_contorta.gene_id62647.merge.bed"

In [4]:
BED = pd.read_csv(bed_file, sep="\t", header=None, names=["chr", "start", "end", "id",  "strand"])
BED_gene = pd.read_csv(bed_file_gene, sep="\t", header=None, names=["chr", "start", "end", "id", "length", "strand", "unk1","unk2","unk3","unk4", "unk5","unk6", "type"])
BED_merge_gene = pd.read_csv(bed_file_gene_merge, sep="\t", header=None, names=["chr", "start", "end", "id",  "strand"])
splicing = pd.DataFrame(BED[[len(i.split(",")) > 1 for i in BED.id.values]])

In [5]:
BED_merge_gene[[len(i.split(",")) > 1 for i in BED_merge_gene.id.values]]

,chr,start,end,id,strand
108,Contig00040_ERROPOS1823190+,39954,58487,"EVM0013757,EVM0016346","+,-"
158,Contig00074_ERROPOS900000+,421336,430705,"EVM0006621,EVM0005342","-,-"
335,Contig00129,2955,6296,"EVM0006792,EVM0001574","+,-"
343,Contig00129,110149,113017,"EVM0007303,EVM0002602","+,+"
344,Contig00129,122393,124605,"EVM0015158,EVM0001919","+,+"
...,...,...,...,...,...
17627,LG07,16388077,16403022,"EVM0012852,EVM0012316","+,+"
17729,LG07,17647671,17661967,"EVM0017707,EVM0014638,EVM0005744","+,+,+"
17746,LG07,17778201,17809086,"EVM0007606,EVM0008642","+,-"
17915,LG07,19063290,19070471,"EVM0006931,EVM0002432","+,+"


In [6]:
records = SeqIO.parse(prot_file, "fasta")
sequences = []
for rec in records:
    seq = SeqRecord(rec.seq,id=rec.description.split("||")[4],name="",description="")
    sequences.append(seq)

In [7]:
records_dedup = SeqIO.parse(dedup_prot_file, "fasta")
sequences_dedup = []
for rec in records_dedup:
    seq = SeqRecord(rec.seq,id=rec.id,name="",description="")
    sequences_dedup.append(seq)

In [135]:
with open(path+"Infer_Anth/data/aristo_contorta/aco.all.pep.fasta", "w") as output_handle:
    SeqIO.write(sequences, output_handle, "fasta")

In [9]:
len(BED_gene)

18315

In [10]:
sequences_dict = {rec.id : rec.seq for rec in sequences}

In [11]:
##retrieve all the id that are not .1 = 1015 sequences

splicing = []
for rec in sequences_dedup:
    if rec.id.split(".")[1] != "1":
        splicing.append(rec)

In [32]:
##among those 1015 sequences, some are the same length as the .1 or shorter but some are longer. 
#We need to identify the .2 and .3 that are longer. and keep only the .1 that are longer than the other transcripts. 

different_length = []
longer_seq = []
shorter_seq = []
for rec in splicing: 
    id_query = rec.id.split(".")[0]
    if len(rec.seq) != len(sequences_dict[id_query+".1"]):
        different_length.append(rec)
        if len(rec.seq) > len(sequences_dict[id_query+".1"]):
            longer_seq.append(rec)
            shorter_seq.append(id_query+".1")

print(len(set(shorter_seq))) ##for 108 .1 the transcripts are longer  

108


In [200]:
for i in range(len(longer_seq)):
    if len(longer_seq[i].seq) > 10+ len(sequences_dict[longer_seq[i].id.split(".")[0]+".1"]):
        print(longer_seq[i].id, len(longer_seq[i].seq), len(sequences_dict[longer_seq[i].id.split(".")[0]+".1"]))

EVM0000243.2 625 593
EVM0000636.2 648 526
EVM0001261.2 457 427
EVM0001340.2 868 849
EVM0001615.2 1017 830
EVM0002203.2 252 230
EVM0002705.2 406 382
EVM0002943.2 256 216
EVM0003008.2 419 387
EVM0003338.2 466 446
EVM0003802.2 265 238
EVM0004177.2 534 490
EVM0004179.2 518 483
EVM0004365.2 285 265
EVM0004365.3 404 265
EVM0004425.2 511 497
EVM0004840.2 473 461
EVM0005221.2 247 207
EVM0005223.2 408 378
EVM0005316.2 279 228
EVM0005322.2 354 334
EVM0005582.2 325 296
EVM0006175.2 206 187
EVM0006178.2 603 373
EVM0006547.2 568 526
EVM0006915.2 400 367
EVM0007457.2 451 423
EVM0007531.2 524 505
EVM0007667.2 608 584
EVM0007667.3 768 584
EVM0007886.2 880 665
EVM0008385.2 543 453
EVM0008469.3 437 417
EVM0008912.2 468 435
EVM0008914.2 546 495
EVM0009289.2 338 299
EVM0009302.2 247 220
EVM0009438.2 478 462
EVM0009845.2 475 452
EVM0009904.2 531 518
EVM0010630.2 471 402
EVM0011265.2 1080 1026
EVM0011329.2 607 485
EVM0011420.2 292 265
EVM0012748.2 883 850
EVM0013291.2 259 247
EVM0013376.2 434 413
EVM0013857

In [33]:
len(sequences_dedup)

19330

In [34]:
longest_fasta = []
for seq in sequences_dedup: ###19330 we removed sequences with same name and same sequence. 
    if (seq.id[-1] == "1") and (seq.id not in shorter_seq):
        longest_fasta.append(seq)

In [35]:
len(longest_fasta)

18207

In [52]:
#now we need to add the longest transcript. .2 or .3... 

#for the shortest seq, retrieve the corresponding transcripts. 
#create a dict of transcritps


shortest_transcripts_dict = {i:[] for i in set(shorter_seq)}


for rec in longer_seq:
    query = rec.id.split(".")[0]+".1"
    if query in shortest_transcripts_dict.keys():
        shortest_transcripts_dict[query].append((rec.id, len(rec.seq)))


In [64]:
for i in shortest_transcripts_dict:
    if len(shortest_transcripts_dict[i]) == 1:
        res = shortest_transcripts_dict[i][0][0]
        rec = SeqRecord(seq=sequences_dict[res], id = res, name="", description="")
        longest_fasta.append(rec)
    else :
        res1 = shortest_transcripts_dict[i][0][0]
        res2 = shortest_transcripts_dict[i][1][0]
        seq1 = sequences_dict[res1]
        seq2 = sequences_dict[res2]
        if len(seq1) >= len(seq2):
            rec = SeqRecord(seq=sequences_dict[res1], id = res1, name="", description="")
        else:
            rec = SeqRecord(seq=sequences_dict[res2], id = res2, name="", description="")
        longest_fasta.append(rec)



In [66]:
len(longest_fasta) == len(BED_gene)

True

In [70]:
longest_fasta_clean = []
for rec in longest_fasta:
    longest_fasta_clean.append(SeqRecord(seq=rec.seq, id=rec.id.split(".")[0], name = "", description=""))

In [72]:
with open(path+"Infer_Anth/data/aristo_contorta/aco.nosplicing.pep.fasta", "w") as output_handle:
    SeqIO.write(longest_fasta_clean, output_handle, "fasta")

In [79]:
BED_gene["chr"] = BED_gene.chr.apply(lambda x: x.split("_")[0] if "_" in x else x)
BED_gene["names"] = BED_gene.id
BED_gene["seqnames"] = BED_gene.chr.apply(lambda x: "aco_" + str(x)) 
BED_gene["ranges"] = [str(i)+"-"+str(j) for i,j in zip(BED_gene["start"], BED_gene["end"])]
BED_gene["gene_id"] = BED_gene["names"]
BED_gene["phase"] = BED_gene["strand"]
BED_gene["source"] = ["coge"]*len(BED_gene)

,seqnames,strand,source,type,phase,names,gene_id,start,end
0,aco_Contig00003,+,coge,gene,+,EVM0018050,EVM0018050,18788,22060
1,aco_Contig00003,-,coge,gene,-,EVM0000565,EVM0000565,22423,23572
2,aco_Contig00003,+,coge,gene,+,EVM0008011,EVM0008011,24902,28150
3,aco_Contig00003,+,coge,gene,+,EVM0005647,EVM0005647,29136,31753
4,aco_Contig00003,+,coge,gene,+,EVM0003460,EVM0003460,32406,35048
...,...,...,...,...,...,...,...,...,...
18310,aco_LG07,-,coge,gene,-,EVM0002131,EVM0002131,20732641,20735977
18311,aco_LG07,-,coge,gene,-,EVM0009627,EVM0009627,20737253,20782594
18312,aco_LG07,+,coge,gene,+,EVM0015372,EVM0015372,20786713,20789068
18313,aco_LG07,+,coge,gene,+,EVM0014580,EVM0014580,20791440,20793170


In [85]:
subset = pd.DataFrame(BED_gene[["seqnames", "strand", "source", "type", "phase", "names", "gene_id", "start", "end"]])
subset.sort_values(by=["seqnames", "start"], inplace=True)
subset.to_csv(path+"Infer_Anth/data/aristo_contorta/aristo_contorta_no_splicing.tsv", sep="\t", index=None)

In [94]:
####where are the yabby in contorta and annona


blast_table = pd.read_csv(path+"Infer_Anth/data/inter_species/ambo_aristo_vitis_cycas_annona_contorta/blast_table.tsv", sep="\t")


In [98]:

yabby_afi = list(["afi_afi_Af03G134500", "afi_afi_Af01G026600", "afi_afi_Af02G072800", "afi_afi_Af05G185300","afi_afi_Af07G024800"])
#INO, YAB5, FIL, YAB2, CRC
yabby_atr = list(["atr_atr_94506842", "atr_atr_94491478", "atr_atr_94503013", "atr_atr_94473924", "atr_atr_94507018"])
#YAB2, CRC, INO, YAB5, FIL
yabby_viv = list(["viv_Vitvi06g00972", "viv_Vitvi02g00510", "viv_Vitvi15g00708", "viv_Vitvi11g00492", "viv_Vitvi01g00013",  "viv_Vitvi08g00274", "viv_viv_Vitvi01g00703"])
# YAB2 FIL FIL YAB5 CRC YAB2 INO
yabby_cycas = list(["cyp_CYCAS_007102", "cyp_CYCAS_008411", "cyp_CYCAS_022386"])


In [110]:
blast_ach = blast_table.loc[blast_table["query"].str.startswith("ach")]

blast_ach[blast_ach.db.isin(yabby_viv)].groupby("query").first()


,db,perc_identity,length,mismatches,gap_open,qstart,qend,tstart,tend,evalue,bitscore
query,,,,,,,,,,,
ach_Anche102Chr2g0051300,viv_Vitvi01g00013,65.0,157,55,0,1,157,9,165,3.400000e-55,211.8
ach_Anche102Chr2g0051650,viv_Vitvi01g00013,65.0,157,55,0,1,157,9,165,3.400000e-55,211.8
ach_Anche102Chr3g0055340,viv_Vitvi06g00972,43.1,153,81,2,14,164,8,156,3.400000e-27,119.0
ach_Anche102Chr3g0055730,viv_Vitvi15g00708,42.8,166,85,3,7,164,10,173,4.500000e-27,118.6
ach_Anche102Chr5g0037200,viv_Vitvi11g00492,61.2,147,38,4,1,144,52,182,3.600000e-40,161.8
ach_Anche102Chr5g0038080,viv_Vitvi11g00492,73.1,182,33,2,1,181,1,167,8.200000e-72,267.3
ach_Anche102Chr5g0038100,viv_Vitvi11g00492,68.4,174,31,3,1,152,1,172,2.000000e-60,229.2
ach_Anche102Scf0005g0001760,viv_Vitvi02g00510,73.7,213,54,2,1,213,1,211,5.900000e-87,317.8
ach_Anche102Scf0009g0001540,viv_Vitvi02g00510,74.5,212,51,3,1,210,1,211,2.300000e-88,322.4


In [116]:
blast_aco = blast_table.loc[blast_table["query"].str.startswith("aco")]
blast_aco[blast_aco.db.isin(yabby_viv)].groupby("query").first()


,db,perc_identity,length,mismatches,gap_open,qstart,qend,tstart,tend,evalue,bitscore
query,,,,,,,,,,,
aco_EVM0000790,viv_Vitvi06g00972,69.0,187,52,4,1,184,1,184,1.900000e-70,262.7
aco_EVM0003345,viv_Vitvi11g00492,82.0,172,26,4,3,172,2,170,9.300000e-75,276.9
aco_EVM0003940,viv_Vitvi02g00510,75.8,211,50,1,1,210,1,211,7.300000e-90,327.4
aco_EVM0005992,viv_Vitvi01g00013,63.4,161,57,1,1,161,9,167,2.400000e-55,212.6
aco_EVM0006221,viv_Vitvi02g00510,73.1,182,48,1,1,181,30,211,7.000000e-73,270.8
aco_EVM0011229,viv_Vitvi15g00708,49.4,162,63,5,7,151,12,171,1.200000e-30,130.6


,query,db,perc_identity,length,mismatches,gap_open,qstart,qend,tstart,tend,evalue,bitscore
779766,aco_EVM0000001,ach_Anche102Chr1g0099590,54.3,269,119,3,1,266,1,268,2.600000e-78,289.7
779767,aco_EVM0000001,ach_Anche102Chr1g0102510,26.3,228,152,5,47,266,20,239,1.300000e-13,74.7
779768,aco_EVM0000001,ach_Anche102Chr1g0102650,26.3,228,152,5,47,266,20,239,1.300000e-13,74.7
779769,aco_EVM0000001,ach_Anche102Chr1g0102680,26.3,228,152,5,47,266,20,239,1.300000e-13,74.7
779770,aco_EVM0000001,ach_Anche102Scf0641g0000030,26.3,228,152,5,47,266,20,239,1.300000e-13,74.7
